# PathSnap — Team 15, Section 5-D
## Colab bootstrap, dataset audit and first map

**Team:** Yash Raghavendra A · PES2UG24CS913 | Deeraj S · PES2UG25CS806 | Ibrahim Khaleel · PES2UG25CS809

This notebook is the starting point for Google Colab. It does **not** train a model or create predictions. It verifies the supplied `Student folder.zip`, produces a reproducible inventory and a first exploratory Folium map.

### Before running
1. In Google Drive, create `MyDrive/PathSnap/data/`.
2. Upload the supplied **Student folder.zip** into that folder.
3. Open this notebook in Colab and save a copy in `MyDrive/PathSnap/notebooks/`.
4. In Colab, choose **Runtime → Change runtime type → T4 GPU** when available. A T4 is not guaranteed on the free tier.
5. Run the cells in order.

Keep the raw dataset out of GitHub unless the organisers explicitly require it. Keep ground-truth `.route` files out of the evaluator UI and prediction payloads.


## 1. Check runtime and GPU
A T4 helps only for code that actually uses CUDA. Many scikit-learn models run on CPU. Record the runtime details for reproducibility.

In [ ]:
import os, sys, platform, subprocess
print('Python:', sys.version)
print('Platform:', platform.platform())
try:
    result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, check=False)
    print(result.stdout if result.returncode == 0 else 'No NVIDIA GPU detected in this runtime.')
except FileNotFoundError:
    print('nvidia-smi is unavailable; this runtime may not have an NVIDIA GPU.')
try:
    import torch
    print('PyTorch:', torch.__version__)
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('PyTorch GPU check unavailable:', exc)


## 2. Install project libraries
The list below covers EDA, classical ML, Bayesian optimisation, maps and later API/UI work. Installation does not imply that every library or GPU is needed for every experiment.

In [ ]:
%pip -q install numpy pandas scipy scikit-learn joblib matplotlib seaborn folium networkx optuna fastapi 'uvicorn[standard]' streamlit requests streamlit-folium pyproj


## 3. Mount Google Drive
Mounting Drive allows this Colab runtime to read the source ZIP and save inventories/maps to your Drive. The data is copied to `/content` for faster repeated reads during this session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_PROJECT = Path('/content/drive/MyDrive/PathSnap')
DATA_ZIP = DRIVE_PROJECT / 'data' / 'Student folder.zip'
OUTPUT_DIR = DRIVE_PROJECT / 'outputs'
LOCAL_DATA_DIR = Path('/content/pathsnap_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print('Expected dataset ZIP:', DATA_ZIP)
print('ZIP exists:', DATA_ZIP.exists())
if not DATA_ZIP.exists():
    raise FileNotFoundError(
        'Upload Student folder.zip to MyDrive/PathSnap/data/ and rerun this cell.'
    )


## 4. Extract the dataset safely
The supplied archive should contain `Student folder/train/` and `Student folder/student_test/`. This cell extracts it to the Colab VM, not into your Git repository.

In [ ]:
import zipfile

if not (LOCAL_DATA_DIR / 'Student folder' / 'train').exists():
    with zipfile.ZipFile(DATA_ZIP, 'r') as zf:
        # Reject path traversal entries before extraction.
        root = LOCAL_DATA_DIR.resolve()
        for member in zf.infolist():
            target = (LOCAL_DATA_DIR / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        zf.extractall(LOCAL_DATA_DIR)

DATA_ROOT = LOCAL_DATA_DIR / 'Student folder'
TRAIN_DIR = DATA_ROOT / 'train'
STUDENT_TEST_DIR = DATA_ROOT / 'student_test'
assert TRAIN_DIR.is_dir(), f'Missing: {TRAIN_DIR}'
assert STUDENT_TEST_DIR.is_dir(), f'Missing: {STUDENT_TEST_DIR}'
print('Extracted to:', DATA_ROOT)
print('Train trip folders:', len([p for p in TRAIN_DIR.iterdir() if p.is_dir()]))
print('Student-test trip folders:', len([p for p in STUDENT_TEST_DIR.iterdir() if p.is_dir()]))


## 5. Read one trip and understand file formats
The raw formats are whitespace-delimited and have no header. `.track` = longitude, latitude, timestamp; `.nodes` = longitude, latitude; `.arcs` = from-node ID, to-node ID; `.route` = chronological reference arc IDs. A GPS row is **not** assumed to correspond to a route row.

In [ ]:
import pandas as pd

def read_trip_files(trip_dir: Path):
    tid = trip_dir.name
    track = pd.read_csv(trip_dir / f'{tid}.track', sep=r'\s+', header=None, names=['longitude','latitude','timestamp'])
    nodes = pd.read_csv(trip_dir / f'{tid}.nodes', sep=r'\s+', header=None, names=['longitude','latitude'])
    arcs = pd.read_csv(trip_dir / f'{tid}.arcs', sep=r'\s+', header=None, names=['from_node_id','to_node_id'])
    route_path = trip_dir / f'{tid}.route'
    route = pd.read_csv(route_path, sep=r'\s+', header=None, names=['arc_id'])['arc_id'].to_numpy() if route_path.exists() else None
    return track, nodes, arcs, route

sample_trip_dir = sorted([p for p in TRAIN_DIR.iterdir() if p.is_dir()])[0]
track, nodes, arcs, route = read_trip_files(sample_trip_dir)
print('Sample trip:', sample_trip_dir.name)
print('GPS track:', track.shape)
print('Nodes:', nodes.shape)
print('Arcs:', arcs.shape)
print('Reference route length:', None if route is None else len(route))
display(track.head())
display(nodes.head())
display(arcs.head())
print('Route arc IDs:', None if route is None else route[:10])


## 6. Create a partition inventory
This records trip-level counts and basic integrity checks. Do not combine observations from one trip across training and testing partitions.

In [ ]:
import numpy as np

def inventory_partition(partition_name: str, partition_dir: Path):
    rows = []
    for trip_dir in sorted(p for p in partition_dir.iterdir() if p.is_dir()):
        tid = trip_dir.name
        track, nodes, arcs, route = read_trip_files(trip_dir)
        gps = track[['longitude','latitude','timestamp']]
        gaps = gps['timestamp'].diff().dropna()
        endpoints = arcs[['from_node_id','to_node_id']].to_numpy(dtype=int)
        invalid_endpoints = int((((endpoints < 0) | (endpoints >= len(nodes))).any(axis=1)).sum()) if len(endpoints) else 0
        invalid_route_ids = None if route is None else int(((route < 0) | (route >= len(arcs))).sum())
        rows.append({
            'partition': partition_name, 'trip_id': tid,
            'gps_observations': len(track), 'nodes': len(nodes), 'arcs': len(arcs),
            'reference_route_edges': None if route is None else len(route),
            'missing_gps_values': int(gps.isna().sum().sum()),
            'nonpositive_time_steps': int((gaps <= 0).sum()),
            'time_gaps_gt_1s': int((gaps > 1).sum()),
            'max_time_gap_seconds': float(gaps.max()) if len(gaps) else 0.0,
            'invalid_arc_endpoints': invalid_endpoints,
            'invalid_route_ids': invalid_route_ids,
        })
    return rows

rows = inventory_partition('train', TRAIN_DIR) + inventory_partition('student_test', STUDENT_TEST_DIR)
inventory = pd.DataFrame(rows)
inventory_path = OUTPUT_DIR / 'trip_inventory_colab.csv'
inventory.to_csv(inventory_path, index=False)
print('Inventory saved:', inventory_path)
display(inventory.groupby('partition').agg(
    trips=('trip_id','count'), gps_observations=('gps_observations','sum'),
    total_nodes=('nodes','sum'), total_arcs=('arcs','sum')
))
display(inventory.head())


## 7. First Folium map: GPS observations and reference route
This is a visual inspection of one training trip. The reference route is reconstructed from its `.route` arc IDs by looking up each arc's two endpoint nodes. It is not created by aligning route rows to GPS rows.

In [ ]:
import folium

# Re-read the sample trip and convert its route arc IDs into node coordinates.
track, nodes, arcs, route = read_trip_files(sample_trip_dir)
center = [float(track['latitude'].median()), float(track['longitude'].median())]
m = folium.Map(location=center, zoom_start=14, control_scale=True)

gps_latlon = track[['latitude','longitude']].to_numpy().tolist()
gps_layer = folium.FeatureGroup(name='Observed GPS', show=True)
folium.PolyLine(gps_latlon, color='blue', weight=3, opacity=0.75, tooltip='Observed GPS').add_to(gps_layer)
for i, (lat, lon) in enumerate(gps_latlon[::max(1, len(gps_latlon)//250)]):
    folium.CircleMarker([lat, lon], radius=2, color='blue', fill=True, fill_opacity=0.6, tooltip=f'GPS sample {i}').add_to(gps_layer)
gps_layer.add_to(m)

if route is not None:
    route_layer = folium.FeatureGroup(name='Supplied reference route', show=True)
    for arc_id in route:
        if 0 <= int(arc_id) < len(arcs):
            u, v = arcs.iloc[int(arc_id)].astype(int).tolist()
            if 0 <= u < len(nodes) and 0 <= v < len(nodes):
                a = [float(nodes.iloc[u]['latitude']), float(nodes.iloc[u]['longitude'])]
                b = [float(nodes.iloc[v]['latitude']), float(nodes.iloc[v]['longitude'])]
                folium.PolyLine([a, b], color='green', weight=4, opacity=0.85).add_to(route_layer)
    route_layer.add_to(m)

folium.LayerControl().add_to(m)
map_path = OUTPUT_DIR / f'eda_map_train_{sample_trip_dir.name}.html'
m.save(map_path)
print('Map saved:', map_path)
m


## 8. Save the notebook outputs and download the inventory
The raw dataset remains in Drive and in this Colab runtime. The notebook, CSV and HTML map should be copied into the team's project repository once reviewed.

In [ ]:
from google.colab import files
files.download(str(inventory_path))
files.download(str(map_path))


## 9. Next implementation milestones
- Implement candidate-road generation using a suitable spatial index and metric/geodesic distances.
- Establish and validate how candidate labels are associated with the supplied reference route; do not assume row-by-row alignment.
- Create trip-grouped train/validation folds.
- Train at least five models and retain configurations, metrics, training time and inference time.
- Tune the selected model using Bayesian optimisation on training-trip folds.
- Implement and compare classifier-only predictions with HMM/Viterbi decoding.
- Export the exact inference pipeline and artifacts needed by the FastAPI server.

**Partition discipline:** 80 training trips for fitting and group-aware CV; 10 student-testing trips for development-stage evaluation; 10 reserved evaluator trips only for final evaluation. Never use evaluator labels for feature selection, tuning or manual corrections.